In [0]:
dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

environment = dbutils.widgets.get("environment").lower()

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "storage_account": "stcentralusjrdev",
        "catalog": "salesjson_dev"
    },
    "prod": {
        "storage_account": "stcentralusjrprod",
        "catalog": "salesjson_prod"
    }
}

env = config[environment]

storage_account = env["storage_account"]
catalog = env["catalog"]

bronze_table = f"{catalog}.bronze.orders_raw"
silver_table = f"{catalog}.silver.orders"
rejected_table = f"{catalog}.silver.rejected_orders"

daily_table = f"{catalog}.gold.daily_sales_summary"
category_table = f"{catalog}.gold.category_sales_summary"
customer_table = f"{catalog}.gold.customer_sales_summary"

checkpoint_path = (
    f"abfss://streaming@{storage_account}.dfs.core.windows.net/"
    "salesjson/checkpoints/bronze_orders/"
)

print("=" * 60)
print("SALES JSON - PHASE VALIDATION")
print("=" * 60)
print(f"Environment     : {environment}")
print(f"Catalog         : {catalog}")
print(f"Storage account : {storage_account}")
print("=" * 60)

In [0]:

bronze_count = spark.table(bronze_table).count()
silver_count = spark.table(silver_table).count()
rejected_count = spark.table(rejected_table).count()

daily_count = spark.table(daily_table).count()
category_count = spark.table(category_table).count()
customer_count = spark.table(customer_table).count()

print("=" * 60)
print("MEDALLION LAYER SUMMARY")
print("=" * 60)
print(f"Bronze records          : {bronze_count}")
print(f"Silver valid records    : {silver_count}")
print(f"Silver rejected records : {rejected_count}")
print(f"Gold daily rows         : {daily_count}")
print(f"Gold category rows      : {category_count}")
print(f"Gold customer rows      : {customer_count}")
print("=" * 60)

In [0]:


display(
    spark.sql(f"""
        SELECT
            source_file,
            COUNT(*) AS rows_ingested,
            MIN(ingestion_timestamp) AS first_ingestion_timestamp,
            MAX(ingestion_timestamp) AS last_ingestion_timestamp
        FROM {bronze_table}
        GROUP BY source_file
        ORDER BY source_file
    """)
)

In [0]:
bronze_columns = [
    field.name
    for field in spark.table(bronze_table).schema.fields
]

print("Bronze columns:")
for column_name in bronze_columns:
    print(f" - {column_name}")

if "shipping_priority" in bronze_columns:
    print("\nPASS - shipping_priority was added through schema evolution.")
else:
    print("\nFAIL - shipping_priority was not found.")

In [0]:
display(
    spark.table(bronze_table)
        .filter("shipping_priority IS NOT NULL")
        .select(
            "order_id",
            "source_file",
            "shipping_priority"
        )
        .orderBy("order_id")
)

In [0]:


display(
    spark.sql(f"""
        SELECT *
        FROM cloud_files_state('{checkpoint_path}')
        ORDER BY path
    """)
)

In [0]:

display(
    spark.table(rejected_table)
        .select(
            "order_id",
            "customer_id",
            "quantity",
            "unit_price",
            "discount",
            "rejection_reason",
            "source_file"
        )
        .orderBy("order_id")
)

display(
    spark.table(rejected_table)
        .groupBy("rejection_reason")
        .count()
        .orderBy("rejection_reason")
)

In [0]:
spark.table(silver_table).printSchema()

In [0]:
display(
    spark.table(daily_table)
        .orderBy("sale_date")
)

display(
    spark.table(category_table)
        .orderBy("net_revenue", ascending=False)
)

display(
    spark.table(customer_table)
        .orderBy("total_spent", ascending=False)
        .limit(20)
)

In [0]:
from pyspark.sql.functions import sum, round


silver_revenue = (
    spark.table(silver_table)
        .agg(
            round(
                sum("net_amount"),
                2
            ).alias("revenue")
        )
        .first()["revenue"]
)

gold_revenue = (
    spark.table(daily_table)
        .agg(
            round(
                sum("net_revenue"),
                2
            ).alias("revenue")
        )
        .first()["revenue"]
)

print("=" * 60)
print("REVENUE RECONCILIATION")
print("=" * 60)
print(f"Silver net revenue : {silver_revenue}")
print(f"Gold net revenue   : {gold_revenue}")

if silver_revenue == gold_revenue:
    print("PASS - Revenue reconciliation successful.")
else:
    print("FAIL - Revenue reconciliation mismatch.")

print("=" * 60)